<a href="https://colab.research.google.com/github/bijelic02/myRAGProject/blob/main/MyRAGProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase **1**
Environment & Data Setup



Connect to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("Mounted successfully")

Mounted at /content/drive
Mounted successfully


This mounts your Google Drive into Colab. When you run it, it asks for permission once and then your Drive is accessible at `/content/drive/MyDrive/`. Your notebook file itself (`.ipynb`) gets saved to Drive automatically if you set it up right.

**Save the notebook itself to Drive**

In Colab go to:
```
File → Save a copy in Drive
```

This moves your notebook from Colab's temporary storage into your actual Google Drive. From that point on, `Ctrl+S` saves it there. Never work from the default "Colab Notebooks" auto-created file without doing this first.
---

**Also save to GitHub (highly recommended)**

The spec specifically asks for a GitHub repo as a deliverable anyway, so set it up early:
```
File → Save a copy in GitHub


Install dependencies

In [2]:
!pip install transformers torch accelerate bitsandbytes \
             sentence-transformers \
             faiss-cpu chromadb \
             rank-bm25 \
             langchain langchain-community nltk \
             gradio \
             pypdf beautifulsoup4 requests \
             wikipedia-api \
             pandas numpy tqdm scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17

Check if everything is actually installed correctly

In [3]:
import transformers
import torch
import sentence_transformers
import faiss
import chromadb
import gradio
import langchain
from rank_bm25 import BM25Okapi
import pandas as pd
import sklearn
import wikipediaapi
import json
import os
from datetime import datetime

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
print("faiss version: OK")
print("chromadb:", chromadb.__version__)
print("gradio:", gradio.__version__)
print("langchain:", langchain.__version__)
print("pandas:", pd.__version__)
print("GPU available:", torch.cuda.is_available())

transformers: 5.0.0
torch: 2.10.0+cu128
sentence_transformers: 5.3.0
faiss version: OK
chromadb: 1.5.5
gradio: 5.50.0
langchain: 1.2.12
pandas: 2.2.2
GPU available: True


Define documents list with metadata

In [5]:
DOCUMENTS = [
    {
        "title": "Rodent",
        "url": "https://en.wikipedia.org/wiki/Rodent",
        "category": "overview",
        "topic": "rodents"
    },
    {
        "title": "List of rodents",
        "url": "https://en.wikipedia.org/wiki/List_of_rodents",
        "category": "overview",
        "topic": "rodents"
    },
    {
        "title": "Beaver",
        "url": "https://en.wikipedia.org/wiki/Beaver",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Nutria",
        "url": "https://en.wikipedia.org/wiki/Nutria",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Gopher",
        "url": "https://en.wikipedia.org/wiki/Gopher",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Chinchilla",
        "url": "https://en.wikipedia.org/wiki/Chinchilla",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Squirrel",
        "url": "https://en.wikipedia.org/wiki/Squirrel",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Capybara",
        "url": "https://en.wikipedia.org/wiki/Capybara",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Guinea pig",
        "url": "https://en.wikipedia.org/wiki/Guinea_pig",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Marmot",
        "url": "https://en.wikipedia.org/wiki/Marmot",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Pika",
        "url": "https://en.wikipedia.org/wiki/Pika",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Rabbit",
        "url": "https://en.wikipedia.org/wiki/Rabbit",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Rat",
        "url": "https://en.wikipedia.org/wiki/Rat",
        "category": "species",
        "topic": "rodents"
    },
]

print(f"Defined {len(DOCUMENTS)} documents")

Defined 13 documents


Fetch and normalize documents

In [6]:
def fetch_wikipedia_page(wiki, title):
    """Fetch a single Wikipedia page and return its text."""
    page = wiki.page(title)
    if not page.exists():
        print(f"  WARNING: Page '{title}' not found")
        return None
    return page.text

In [7]:
def normalize_text(text):
    """Basic text normalization."""
    # Remove excessive newlines
    lines = text.split('\n')
    # Filter out very short lines (usually section headers or empty)
    lines = [line.strip() for line in lines if len(line.strip()) > 30]
    return ' '.join(lines)

In [8]:
def ingest_documents(document_list, save_path):
    """Fetch all documents from Wikipedia and save with metadata."""

    # Initialize Wikipedia API
    wiki = wikipediaapi.Wikipedia(
        language='en',
        user_agent='RodentRAGProject/1.0'
    )

    ingested = []
    failed = []

    for doc in document_list:
        print(f"Fetching: {doc['title']}...")

        try:
            raw_text = fetch_wikipedia_page(wiki, doc['title'])

            if raw_text is None:
                failed.append(doc['title'])
                continue

            clean_text = normalize_text(raw_text)

            # Build the full document object with metadata
            document = {
                "id": doc['title'].lower().replace(' ', '_'),
                "title": doc['title'],
                "text": clean_text,
                "metadata": {
                    "source": doc['url'],
                    "category": doc['category'],
                    "topic": doc['topic'],
                    "date_ingested": datetime.now().strftime("%Y-%m-%d"),
                    "char_count": len(clean_text),
                    "word_count": len(clean_text.split())
                }
            }

            ingested.append(document)
            print(f"  OK — {document['metadata']['word_count']} words")

        except Exception as e:
            print(f"  FAILED: {e}")
            failed.append(doc['title'])

    # Save to JSON file on Drive
    os.makedirs(save_path, exist_ok=True)
    output_file = os.path.join(save_path, 'documents.json')

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(ingested, f, indent=2, ensure_ascii=False)

    print(f"\n{'='*40}")
    print(f"Successfully ingested: {len(ingested)} documents")
    print(f"Failed: {len(failed)} documents {failed if failed else ''}")
    print(f"Saved to: {output_file}")

    return ingested



In [6]:
# Run ingestion - saves to Google Drive
DOCS_SAVE_PATH = "/content/drive/MyDrive/rag_project/documents"


In [10]:
documents = ingest_documents(DOCUMENTS, DOCS_SAVE_PATH)

Fetching: Rodent...
  OK — 6977 words
Fetching: List of rodents...
  OK — 2111 words
Fetching: Beaver...
  OK — 5947 words
Fetching: Nutria...
  OK — 4645 words
Fetching: Gopher...
  OK — 1169 words
Fetching: Chinchilla...
  OK — 1605 words
Fetching: Squirrel...
  OK — 1979 words
Fetching: Capybara...
  OK — 2160 words
Fetching: Guinea pig...
  OK — 7287 words
Fetching: Marmot...
  OK — 692 words
Fetching: Pika...
  OK — 2107 words
Fetching: Rabbit...
  OK — 6555 words
Fetching: Rat...
  OK — 4628 words

Successfully ingested: 13 documents
Failed: 0 documents 
Saved to: /content/drive/MyDrive/rag_project/documents/documents.json


Preview what you fetched

In [11]:
def preview_documents(docs):
    """Print a summary table of all ingested documents."""

    rows = []
    for doc in docs:
        rows.append({
            "ID": doc['id'],
            "Title": doc['title'],
            "Category": doc['metadata']['category'],
            "Words": doc['metadata']['word_count'],
            "Chars": doc['metadata']['char_count'],
            "Date Ingested": doc['metadata']['date_ingested']
        })

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print(f"\nTotal words across all documents: {df['Words'].sum():,}")
    print(f"Average words per document: {df['Words'].mean():.0f}")

In [12]:


preview_documents(documents)

             ID           Title Category  Words  Chars Date Ingested
         rodent          Rodent overview   6977  44898    2026-03-31
list_of_rodents List of rodents overview   2111  14286    2026-03-31
         beaver          Beaver  species   5947  36695    2026-03-31
         nutria          Nutria  species   4645  29168    2026-03-31
         gopher          Gopher  species   1169   7577    2026-03-31
     chinchilla      Chinchilla  species   1605  10184    2026-03-31
       squirrel        Squirrel  species   1979  12876    2026-03-31
       capybara        Capybara  species   2160  13378    2026-03-31
     guinea_pig      Guinea pig  species   7287  45021    2026-03-31
         marmot          Marmot  species    692   4338    2026-03-31
           pika            Pika  species   2107  14352    2026-03-31
         rabbit          Rabbit  species   6555  40597    2026-03-31
            rat             Rat  species   4628  29119    2026-03-31

Total words across all documents:

Load documents back from Drive (for future sessions)

In [7]:
def load_documents(save_path):
    """Load previously ingested documents from Drive."""
    output_file = os.path.join(save_path, 'documents.json')

    if not os.path.exists(output_file):
        print("No saved documents found. Run ingestion first.")
        return []

    with open(output_file, 'r', encoding='utf-8') as f:
        docs = json.load(f)

    print(f"Loaded {len(docs)} documents from {output_file}")
    return docs

# In future sessions, instead of re-fetching from Wikipedia,
# just call this:
# documents = load_documents(DOCS_SAVE_PATH)

# Phase **2**
Three types of chunking



First - we want to load our files from drive

In [ ]:
# This is a copy of code from phase 1
DOCS_SAVE_PATH = "/content/drive/MyDrive/rag_project/documents"


In [ ]:
def load_documents(save_path):
    """Load previously ingested documents from Drive."""
    output_file = os.path.join(save_path, 'documents.json')

    if not os.path.exists(output_file):
        print("No saved documents found. Run ingestion first.")
        return []

    with open(output_file, 'r', encoding='utf-8') as f:
        docs = json.load(f)

    print(f"Loaded {len(docs)} documents from {output_file}")
    return docs


In [8]:
documents = load_documents(DOCS_SAVE_PATH)

Loaded 13 documents from /content/drive/MyDrive/rag_project/documents/documents.json
